### Trends
This notebook aims to provide an interactive visualization of trips duration, length, and frequency over time, differentiating trends by operator.

Use `panel serve trends_dashboard.ipynb tratte_forti_by_hour.py` on anaconda prompt to visualize the dashboards showing trends and most travelled routes.

### Imports

In [ ]:
import pandas as pd
import numpy as np
import panel as pn
pn.extension('tabulator')

import hvplot.pandas

In [18]:
df = pd.read_parquet('C:/Users/luisa/Desktop/escooter_trento/data/tripsv3_cleaned.parquet') 

### Preprocessing

In [19]:
df['trip_start'] = [df['trip_start'][i].replace(tzinfo=None) for i in range(len(df))]
df['trip_end'] = [df['trip_end'][i].replace(tzinfo=None) for i in range(len(df))]
df['trip_start'] =  pd.to_datetime(df['trip_start'], format='%Y-%m-%d %H:%M:%S')
# add trip duration (in seconds)
df['trip_duration_minutes'] = [(df.trip_end[i]-df.trip_start[i]).total_seconds()/60 for i in range(len(df))]
# date only
df['date'] = pd.to_datetime(df['trip_start']).dt.date

In [20]:
number_trips = df.groupby(['date', 'operator'])['date'].agg(["count"]).reset_index()
df = df[['date', 'trip_length', 'trip_duration_minutes', 'operator']]
df = df.groupby(['date','operator']).agg({'trip_length': ['sum'],'trip_duration_minutes':['sum']}).reset_index()
df.columns = df.columns.droplevel(1)
df = pd.merge(df, number_trips, on=['date','operator'])
df = df.rename(columns={'count':'number_trips'})

In [ ]:
df['year'] = [df.at[i,'date'].year for i in range(len(df))]

In [21]:
# Make DataFrame Pipeline Interactive
idf = df.interactive()

### Trends

In [22]:
# Slider

# import datetime as dt
# date_range_slider = pn.widgets.DateRangeSlider(
#     name='Date Range Slider',
#     start=min(df.date), end=max(df.date),
#     value=(dt.datetime(2021, 1, 1), dt.datetime(2022, 1, 1)),
# )

# date_range_slider

year_slider = pn.widgets.IntSlider(name='Show data up to year', start=2020, end=2022, step=1, value=2021)
year_slider

BokehModel(combine_events=True, render_bundle={'docs_json': {'7dd0a007-36b9-47a2-b7dd-46420f385abe': {'defs': …

IntSlider(end=2022, name='Show data up to year', start=2020, value=2021)

In [ ]:
# Radio buttons 
yaxis = pn.widgets.RadioButtonGroup(
    name='Y axis', 
    options=['trip_duration_minutes', 'trip_length','number_trips'],
    button_type='success'
)

In [ ]:
trip_pipeline = (
    idf[(idf.year <= year_slider)] #[(idf.date <= end_date)&(idf.date >= start_date)]
    .groupby(['date', 'operator'])[yaxis].mean()
    .to_frame()
    .reset_index()
    .sort_values(by='date')  
    .reset_index(drop=True)
)

In [ ]:
trip_pipeline

BokehModel(combine_events=True, render_bundle={'docs_json': {'484341a4-803c-47ba-a7f0-ddc0a42f12b7': {'defs': …

In [ ]:
trip_plot = trip_pipeline.hvplot(x='date', by='operator', y=yaxis, line_width=2, title="Trips over time by operator")
trip_plot

BokehModel(combine_events=True, render_bundle={'docs_json': {'ea9f2a12-2232-4acc-acf2-aabb10d901b6': {'defs': …

## DataTable

In [ ]:
data_table = trip_pipeline.pipe(pn.widgets.Tabulator, pagination='remote', page_size = 10, sizing_mode='stretch_width') 
data_table

BokehModel(combine_events=True, render_bundle={'docs_json': {'d3a47435-75fe-4fa7-8e59-9ae8089ea5a6': {'defs': …

## Dashboard

In [ ]:
#Layout using Template
template = pn.template.FastListTemplate(
    title='Trips over time', 
    sidebar=[pn.pane.Markdown("# Trends"), 
             pn.pane.Markdown("#### Visualize trips duration (cumulative times of all trips by day, by operator), length (cumulative by day by operator, in meters), and number of trips over time by day, differentiated by operator."),
             pn.pane.Markdown("## Settings"),
             year_slider],
    main=[pn.Row(pn.Column(yaxis, 
                           trip_plot.panel(width=700), margin=(0,25)), 
                 data_table.panel(width=500))
         ],
    accent_base_color="#88d8b0",
    header_background="#88d8b0",
)

template.servable();

In [ ]:
# template.show()

Launching server at http://localhost:51368
